# 📓 Notebook 2｜用 t-test 選特徵（站 2）—— 重現課本 Example 5.3

> 對應講義 Part 2（5.4）。一個特徵在兩類的平均值差多少才算「真的不同」？
> 用 t-test 檢定 $\mu_1-\mu_2$ 是否顯著非零。顯著就留，不顯著就丟。

In [ ]:
import numpy as np
from scipy import stats

# 課本 Example 5.3 的兩類樣本
w1 = np.array([3.5, 3.7, 3.9, 4.1, 3.4, 3.5, 4.1, 3.8, 3.6, 3.7])
w2 = np.array([3.2, 3.6, 3.1, 3.4, 3.0, 3.4, 2.8, 3.1, 3.3, 3.6])
N = len(w1)
print('ω1 平均:', round(w1.mean(), 2), ' ω2 平均:', round(w2.mean(), 2))

### 課本公式

$$s_z^2=\frac12(\hat\sigma_1^2+\hat\sigma_2^2),\qquad q=\frac{\bar x-\bar y}{s_z\sqrt{2/N}}$$

課本算得 $\bar x=3.73,\ \hat\sigma_1^2=0.0601,\ \bar y=3.25,\ \hat\sigma_2^2=0.0672,\ q=4.25$。

In [ ]:
# 親手重現課本數值
xbar, ybar = w1.mean(), w2.mean()
s1 = w1.var(ddof=1); s2 = w2.var(ddof=1)   # 不偏變異數
sz2 = 0.5 * (s1 + s2)
q = (xbar - ybar) / (np.sqrt(sz2) * np.sqrt(2 / N))
print(f'x̄={xbar:.2f}  σ̂1²={s1:.4f}  ȳ={ybar:.2f}  σ̂2²={s2:.4f}')
print(f'q = {q:.2f}   （課本 4.25）')

### 對照接受區間，判定「選中/丟棄」

自由度 $2N-2=18$、顯著水準 0.05 的接受區間是 $[-2.10, 2.10]$（課本 Table 5.2）。
$q=4.25$ 落在區間外 → 拒絕 $H_0$ → 兩類均值顯著不同 → **特徵選中**。

In [ ]:
# 用 scipy 查 t 分布的臨界值（雙尾 0.05）
df = 2 * N - 2
tcrit = stats.t.ppf(0.975, df)   # 0.975 分位 = 雙尾 0.05 的臨界值
print(f'自由度 {df}，臨界值 ±{tcrit:.2f}（課本 2.10）')
print('判定:', '選中 ✓（顯著不同）' if abs(q) > tcrit else '丟棄 ✗（不顯著）')

### 補充：課本 Example 5.2 的信賴區間

課本 5.4.1 的 Example 5.2：$N=16$（自由度 15）、$\hat\sigma=0.23$、顯著水準 $\rho=0.025$，
信賴區間為 $1.207<\hat\mu<1.493$。用 scipy 重現。

In [ ]:
# 重現 Example 5.2 的信賴區間
N2, df2, s_hat, mu_hat = 16, 15, 0.23, 1.35   # mu_hat 取區間中點
tcrit2 = stats.t.ppf(0.975, df2)              # 雙尾 0.05 → 0.975 分位
half = tcrit2 * s_hat / np.sqrt(N2)
print(f't 臨界值 = {tcrit2:.3f}（課本 Table 5.2: 15 自由度 0.975 → 2.13）')
print(f'信賴區間 = [{mu_hat - half:.3f}, {mu_hat + half:.3f}]（課本 1.207 < μ̂ < 1.493）')

### 對多個特徵逐一檢定（實戰用法）

把上面的流程包成函式，對每個特徵算 q 值，丟掉不顯著的。

In [ ]:
def t_test_feature(x1, x2, alpha=0.05):
    N = len(x1)
    xbar, ybar = x1.mean(), x2.mean()
    sz2 = 0.5 * (x1.var(ddof=1) + x2.var(ddof=1))
    q = (xbar - ybar) / (np.sqrt(sz2) * np.sqrt(2 / N))
    tcrit = stats.t.ppf(1 - alpha / 2, 2 * N - 2)
    return q, abs(q) > tcrit

# 合成 5 個特徵：前 3 個有區辨力，後 2 個是雜訊
rng = np.random.default_rng(1)
for i in range(5):
    gap = 1.0 if i < 3 else 0.0          # 後兩個無區辨力
    a = rng.normal(0, 1, 10); b = rng.normal(gap, 1, 10)
    q, keep = t_test_feature(a, b)
    print(f'特徵 {i}: q={q:+.2f}  →  {"選中" if keep else "丟棄"}')

# ✏️ 練習：把 gap 改成 0.5，看 q 值如何變化
